# TP4 - Online et stochastique pour les CNN

Ce notebook couvre les parties TP4 du mini-projet:

- **TP4-OGD**: gradient en ligne pour la regression CNN;
- **TP4-OSD**: sous-gradient en ligne pour la classification binaire CNN;
- **TP4-SGD**: gradient stochastique en regression;
- **TP4-SSG**: sous-gradient stochastique en classification.


## Objectif TP4

Le TP4 demande de passer d'un cadre purement batch a un cadre plus sequentiel:

- en **online**, les exemples arrivent un par un et le modele est mis a jour apres chaque observation;
- en **stochastique**, on compare differentes granularites de mise a jour, de pas d'apprentissage et d'ordre de parcours des donnees.

Le point central est de relier ces variantes au meme cadre CNN que dans les TP precedents, pour la regression et pour la classification binaire.


In [ ]:
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass

from data_loader import create_dataloaders, resolve_default_paths
from experiment_spec import EXPERIMENTAL_FRAME
from train_common import (
    BinaryHingeLoss,
    build_model,
    evaluate_classification,
    evaluate_regression,
    select_device,
    set_seed,
    train_one_epoch,
)


## Cadre experimental

On garde le meme cadre que dans les autres notebooks:

- dataset: `CelebA reduced`;
- regression: somme des attributs;
- classification binaire: `Smiling`, labels `{-1, +1}`, decision `sign(score)`, perte hinge;
- architectures: `CNN1 = simple`, `CNN2 = improved`.

Le meme split est conserve afin de comparer loyalement les variantes online et stochastiques.


In [ ]:
defaults = resolve_default_paths()
device = select_device('auto')

MODEL_NAMES = ['simple', 'improved']

BASE_CONFIG = {
    'image_dir': defaults['image_dir'],
    'attr_file': defaults['attr_file'],
    'partition_file': defaults['partition_file'],
    'image_size': 64,
    'num_workers': 0,
    'seed': 42,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    'augment': False,
}

ONLINE_LR_REGRESSION = 1e-3
ONLINE_LR_CLASSIFICATION = 1e-3
ONLINE_MAX_STEPS = 128
REFERENCE_EPOCHS = 2
PROJECTION_RADIUS = 25.0

STOCHASTIC_EPOCHS = 4
STOCHASTIC_WEIGHT_DECAY = 1e-4
POINT_BATCH_SIZE = 1
MINI_BATCH_SIZE = 32

for key in ['image_dir', 'attr_file']:
    print(key, '->', BASE_CONFIG[key], Path(BASE_CONFIG[key]).exists())
print('partition_file ->', BASE_CONFIG['partition_file'], Path(BASE_CONFIG['partition_file']).exists())
print('device ->', device)


## Fonctions utilitaires

Les fonctions suivantes definissent:

- un chargeur commun pour recuperer le meme split;
- une reference offline pour approximer le meilleur predicteur fixe;
- une mise a jour online avec projection optionnelle;
- un protocole stochastique permettant de comparer point unique, mini-lot, pas constant, pas decroissant, ordre fixe et ordre aleatoire.


In [ ]:
def get_loaders(task, batch_size):
    common_kwargs = dict(
        img_dir=BASE_CONFIG['image_dir'],
        attr_file=BASE_CONFIG['attr_file'],
        partition_file=BASE_CONFIG['partition_file'] if Path(BASE_CONFIG['partition_file']).exists() else None,
        batch_size=batch_size,
        image_size=BASE_CONFIG['image_size'],
        num_workers=BASE_CONFIG['num_workers'],
        seed=BASE_CONFIG['seed'],
        val_ratio=BASE_CONFIG['val_ratio'],
        test_ratio=BASE_CONFIG['test_ratio'],
        augment=BASE_CONFIG['augment'],
    )
    if task == 'regression':
        return create_dataloaders(target_type='regression', **common_kwargs)
    return create_dataloaders(
        target_type='classification',
        target_column=EXPERIMENTAL_FRAME['classification']['target_column'],
        classification_label_scheme=EXPERIMENTAL_FRAME['classification']['label_scheme'],
        **common_kwargs,
    )


def get_fixed_or_random_train_loader(task, batch_size, shuffle):
    loaders, dataset_sizes = get_loaders(task, batch_size=batch_size)
    train_dataset = loaders['train'].dataset
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=BASE_CONFIG['num_workers'],
        pin_memory=torch.cuda.is_available(),
    )
    loaders['train'] = train_loader
    return loaders, dataset_sizes


def project_parameters_l2(model, radius):
    if radius is None:
        return
    with torch.no_grad():
        total_norm_sq = 0.0
        for parameter in model.parameters():
            total_norm_sq += float(torch.sum(parameter.data ** 2).item())
        total_norm = math.sqrt(total_norm_sq)
        if total_norm <= radius or total_norm == 0.0:
            return
        scale = radius / total_norm
        for parameter in model.parameters():
            parameter.data.mul_(scale)


def build_reference_model(task, model_name, epochs=REFERENCE_EPOCHS):
    set_seed(BASE_CONFIG['seed'])
    loaders, _ = get_loaders(task, batch_size=MINI_BATCH_SIZE)
    model = build_model(model_name, task=task).to(device)

    if task == 'regression':
        criterion = nn.MSELoss()
    else:
        criterion = BinaryHingeLoss()

    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=STOCHASTIC_WEIGHT_DECAY)
    for _ in range(epochs):
        train_one_epoch(model, loaders['train'], criterion, optimizer, device)
    model.eval()
    return model


def run_online_experiment(task, model_name, lr, max_steps, projection_radius=None):
    set_seed(BASE_CONFIG['seed'])
    loaders, dataset_sizes = get_fixed_or_random_train_loader(task, batch_size=1, shuffle=False)
    model = build_model(model_name, task=task).to(device)
    reference_model = build_reference_model(task, model_name)

    if task == 'regression':
        criterion = nn.MSELoss()
        evaluator = evaluate_regression
    else:
        criterion = BinaryHingeLoss()
        evaluator = evaluate_classification

    optimizer = optim.SGD(model.parameters(), lr=lr)
    history = []
    cumulative_loss = 0.0
    cumulative_reference_loss = 0.0

    for step, (inputs, targets) in enumerate(loaders['train'], start=1):
        if step > max_steps:
            break

        inputs = inputs.to(device)
        targets = targets.to(device).view(-1)

        with torch.no_grad():
            reference_outputs = reference_model(inputs).view(-1)
            reference_loss = criterion(reference_outputs, targets).item()

        optimizer.zero_grad()
        outputs = model(inputs).view(-1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        project_parameters_l2(model, projection_radius)

        current_loss = float(loss.item())
        cumulative_loss += current_loss
        cumulative_reference_loss += float(reference_loss)
        regret = cumulative_loss - cumulative_reference_loss

        history.append({
            'step': step,
            'loss': current_loss,
            'reference_loss': float(reference_loss),
            'cumulative_loss': cumulative_loss,
            'cumulative_reference_loss': cumulative_reference_loss,
            'regret': regret,
        })

    val_metrics = evaluator(model, loaders['val'], criterion, device)
    test_metrics = evaluator(model, loaders['test'], criterion, device)
    return {
        'task': task,
        'model': model_name,
        'lr': lr,
        'max_steps': max_steps,
        'projection_radius': projection_radius,
        'dataset_sizes': dataset_sizes,
        'history': pd.DataFrame(history),
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
    }


def run_stochastic_experiment(task, model_name, batch_size, lr_mode, shuffle, epochs, base_lr):
    set_seed(BASE_CONFIG['seed'])
    loaders, dataset_sizes = get_fixed_or_random_train_loader(task, batch_size=batch_size, shuffle=shuffle)
    model = build_model(model_name, task=task).to(device)

    if task == 'regression':
        criterion = nn.MSELoss()
        evaluator = evaluate_regression
    else:
        criterion = BinaryHingeLoss()
        evaluator = evaluate_classification

    optimizer = optim.SGD(model.parameters(), lr=base_lr, weight_decay=STOCHASTIC_WEIGHT_DECAY)
    history = []

    for epoch in range(1, epochs + 1):
        if lr_mode == 'decay':
            current_lr = base_lr / math.sqrt(epoch)
        else:
            current_lr = base_lr
        for group in optimizer.param_groups:
            group['lr'] = current_lr

        train_loss = train_one_epoch(model, loaders['train'], criterion, optimizer, device)
        val_metrics = evaluator(model, loaders['val'], criterion, device)
        row = {
            'epoch': epoch,
            'lr': current_lr,
            'train_loss': train_loss,
            'val_loss': val_metrics['loss'],
        }
        if task == 'regression':
            row.update({
                'val_mae': val_metrics['mae'],
                'val_rmse': val_metrics['rmse'],
                'val_r2': val_metrics['r2'],
            })
        else:
            row.update({
                'val_accuracy': val_metrics['accuracy'],
                'val_precision': val_metrics['precision'],
                'val_recall': val_metrics['recall'],
                'val_f1': val_metrics['f1'],
            })
        history.append(row)

    test_metrics = evaluator(model, loaders['test'], criterion, device)
    return {
        'task': task,
        'model': model_name,
        'batch_size': batch_size,
        'lr_mode': lr_mode,
        'shuffle': shuffle,
        'epochs': epochs,
        'base_lr': base_lr,
        'dataset_sizes': dataset_sizes,
        'history': pd.DataFrame(history),
        'test_metrics': test_metrics,
    }


def summarize_stochastic(results, task):
    rows = []
    for item in results:
        last = item['history'].iloc[-1].to_dict()
        row = {
            'model': item['model'],
            'batch_size': item['batch_size'],
            'lr_mode': item['lr_mode'],
            'order': 'random' if item['shuffle'] else 'fixed',
            'base_lr': item['base_lr'],
            'final_val_loss': last['val_loss'],
            'test_loss': item['test_metrics']['loss'],
        }
        if task == 'regression':
            row.update({
                'final_val_rmse': last['val_rmse'],
                'test_rmse': item['test_metrics']['rmse'],
                'test_r2': item['test_metrics']['r2'],
            })
        else:
            row.update({
                'final_val_f1': last['val_f1'],
                'test_f1': item['test_metrics']['f1'],
                'test_accuracy': item['test_metrics']['accuracy'],
            })
        rows.append(row)
    return pd.DataFrame(rows)


## TP4-OGD - Gradient en ligne pour la regression CNN

On considere une arrivee sequentielle des exemples, un exemple a la fois. A chaque pas:

1. on observe un exemple;
2. on calcule la perte de regression;
3. on met a jour le CNN;
4. on suit la perte cumulative et le regret.

La projection `L2` est incluse de facon optionnelle pour illustrer le point **TP4-OGD.4**.


In [ ]:
online_regression_results = []

for model_name in MODEL_NAMES:
    print(f'[online regression] model={model_name}')
    result = run_online_experiment(
        task='regression',
        model_name=model_name,
        lr=ONLINE_LR_REGRESSION,
        max_steps=ONLINE_MAX_STEPS,
        projection_radius=PROJECTION_RADIUS,
    )
    online_regression_results.append(result)

for item in online_regression_results:
    display(item['history'].head())
    print(item['model'], item['val_metrics'], item['test_metrics'])


In [ ]:
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(14, 4), sharey=False)

for ax, item in zip(axes, online_regression_results):
    history = item['history']
    ax.plot(history['step'], history['cumulative_loss'], label='online cumulative loss')
    ax.plot(history['step'], history['cumulative_reference_loss'], label='reference cumulative loss')
    ax.plot(history['step'], history['regret'], label='regret')
    ax.set_title(f"Online regression - {item['model']}")
    ax.set_xlabel('Step')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()


## TP4-OSD - Sous-gradient en ligne pour la classification CNN

On repete maintenant le meme schema en ligne pour la classification binaire avec perte hinge. On mesure la perte cumulative et le regret par rapport a un predicteur de reference entraine offline sur le meme split.


In [ ]:
online_classification_results = []

for model_name in MODEL_NAMES:
    print(f'[online classification] model={model_name}')
    result = run_online_experiment(
        task='classification',
        model_name=model_name,
        lr=ONLINE_LR_CLASSIFICATION,
        max_steps=ONLINE_MAX_STEPS,
        projection_radius=PROJECTION_RADIUS,
    )
    online_classification_results.append(result)

for item in online_classification_results:
    display(item['history'].head())
    print(item['model'], item['val_metrics'], item['test_metrics'])


In [ ]:
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(14, 4), sharey=False)

for ax, item in zip(axes, online_classification_results):
    history = item['history']
    ax.plot(history['step'], history['cumulative_loss'], label='online cumulative loss')
    ax.plot(history['step'], history['cumulative_reference_loss'], label='reference cumulative loss')
    ax.plot(history['step'], history['regret'], label='regret')
    ax.set_title(f"Online classification - {item['model']}")
    ax.set_xlabel('Step')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()


## TP4-SGD - Gradient stochastique en regression

Le PDF demande de comparer, dans la partie regression:

- **point unique** vs **mini-lot**;
- **pas constant** vs **pas decroissant**;
- **ordre fixe** vs **ordre aleatoire**.

La cellule suivante explore toutes ces combinaisons sur `CNN1` et `CNN2`.


In [ ]:
stochastic_regression_results = []
regression_setups = [
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': True, 'base_lr': 1e-3},
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': True, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': True, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': True, 'base_lr': 1e-3},
]

for model_name in MODEL_NAMES:
    for setup in regression_setups:
        print(f"[stochastic regression] model={model_name} setup={setup}")
        result = run_stochastic_experiment(
            task='regression',
            model_name=model_name,
            epochs=STOCHASTIC_EPOCHS,
            **setup,
        )
        stochastic_regression_results.append(result)

stochastic_regression_summary = summarize_stochastic(stochastic_regression_results, task='regression')
stochastic_regression_summary.sort_values(['model', 'test_rmse']).reset_index(drop=True)


In [ ]:
display(stochastic_regression_summary.sort_values(['model', 'test_rmse']).reset_index(drop=True))

fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(14, 4), sharey=False)

for ax, model_name in zip(axes, MODEL_NAMES):
    subset = stochastic_regression_summary[stochastic_regression_summary['model'] == model_name]
    labels = [f"bs={row.batch_size}|{row.lr_mode}|{row.order}" for row in subset.itertuples()]
    ax.bar(labels, subset['test_rmse'])
    ax.set_title(f'Stochastic regression - {model_name}')
    ax.set_ylabel('Test RMSE')
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()


## TP4-SSG - Sous-gradient stochastique en classification

On fait la meme comparaison en classification binaire avec la perte hinge. Le critere principal de qualite sera ici le `F1` et l'accuracy sur validation et test.


In [ ]:
stochastic_classification_results = []
classification_setups = [
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': True, 'base_lr': 1e-3},
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': POINT_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': True, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'constant', 'shuffle': True, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': False, 'base_lr': 1e-3},
    {'batch_size': MINI_BATCH_SIZE, 'lr_mode': 'decay', 'shuffle': True, 'base_lr': 1e-3},
]

for model_name in MODEL_NAMES:
    for setup in classification_setups:
        print(f"[stochastic classification] model={model_name} setup={setup}")
        result = run_stochastic_experiment(
            task='classification',
            model_name=model_name,
            epochs=STOCHASTIC_EPOCHS,
            **setup,
        )
        stochastic_classification_results.append(result)

stochastic_classification_summary = summarize_stochastic(stochastic_classification_results, task='classification')
stochastic_classification_summary.sort_values(['model', 'test_f1'], ascending=[True, False]).reset_index(drop=True)


In [ ]:
display(stochastic_classification_summary.sort_values(['model', 'test_f1'], ascending=[True, False]).reset_index(drop=True))

fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(14, 4), sharey=False)

for ax, model_name in zip(axes, MODEL_NAMES):
    subset = stochastic_classification_summary[stochastic_classification_summary['model'] == model_name]
    labels = [f"bs={row.batch_size}|{row.lr_mode}|{row.order}" for row in subset.itertuples()]
    ax.bar(labels, subset['test_f1'])
    ax.set_title(f'Stochastic classification - {model_name}')
    ax.set_ylabel('Test F1')
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()


## Interpretation attendue

Lorsque les cellules auront ete executees, il faudra commenter:

1. si l'apprentissage online reste stable ou non sur les premieres arrives d'exemples;
2. si la projection aide a borner les poids et a limiter certaines derives;
3. si le point unique generalise mieux ou moins bien que le mini-lot;
4. si le pas decroissant stabilise mieux que le pas constant;
5. si l'ordre aleatoire aide plus que l'ordre fixe;
6. si `CNN2` est plus sensible que `CNN1` dans le cadre online ou stochastique.


## Synthese TP4

Ce notebook couvre les points du PDF de la facon suivante:

- **TP4-OGD.1**: arrivee des exemples un par un via `batch_size = 1` et ordre fixe;
- **TP4-OGD.2**: mise a jour online via une etape `SGD` apres chaque exemple;
- **TP4-OGD.3**: etude du regret via `cumulative_loss - cumulative_reference_loss`;
- **TP4-OGD.4**: projection eventuelle via `project_parameters_l2`;
- **TP4-OSD.1 a TP4-OSD.3**: meme logique en classification avec perte hinge;
- **TP4-SGD.1 a TP4-SGD.6**: comparaison point unique / mini-lot, pas constant / decroissant, ordre fixe / aleatoire en regression;
- **TP4-SSG.1 a TP4-SSG.6**: meme comparaison en classification binaire.

Le notebook reste compatible avec le reste du projet: meme dataset, memes architectures, memes taches, et memes metriques de validation et de test.
